# The Tokenizer Trap

This notebook demonstrates why tokenizer choice matters for specialized medical and legal domains. A general-purpose tokenizer can split domain terms into many subword fragments, increasing sequence length and the approximate quadratic cost of self-attention.

The experiment compares a pretrained tokenizer with a small BPE tokenizer trained on domain text. The sample corpus is intentionally tiny so the notebook can run inline; meaningful production measurements require a large, representative, legally permitted corpus.

## Interview idea

The risk is not simply that specialized words are unknown. Modern tokenizers can represent unknown words with subwords. The production risk is inefficient fragmentation:

- More tokens create longer sequences.
- Longer sequences consume more memory and reduce useful context.
- Self-attention is approximately $O(n^2)$ in sequence length.

If tokenization makes a sequence four times longer, the attention work can become approximately $4^2 = 16$ times larger. A practical fix is to measure token efficiency and then adapt or retrain the tokenizer, followed by the model training needed for its changed vocabulary.

In [ ]:
# Install the notebook environment from the project root before running:
# uv sync --extra notebook
# %pip install -q ".[notebook]"

Note: you may need to restart the kernel to use updated packages.


c:\Users\berna\OneDrive\Documents\Projects\research-assistant\.venv\Scripts\python.exe: No module named pip


In [ ]:
from pathlib import Path
import re

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from transformers import AutoTokenizer

ModuleNotFoundError: No module named 'tokenizers'

: 

In [ ]:
# A public model keeps the demonstration easy to run. Llama tokenizers
# may require accepting a license and logging in to Hugging Face.
BASE_MODEL = "bert-base-uncased"
DOMAIN_VOCAB_SIZE = 8000
CORPUS_FILE = Path("domain_corpus.txt")

# This fallback lets the notebook run without any additional local file.
# Replace it with a large, representative corpus for useful results.
SAMPLE_CORPUS = """
The patient was diagnosed with glioblastoma after magnetic resonance imaging.
The neurosurgeon explained the aneurysm treatment and postoperative prognosis.
Glioblastoma multiforme can require chemotherapy, radiotherapy, and resection.
The clinical trial evaluated immunotherapy for recurrent glioblastoma.
The subpoena requires production of electronically stored information.
The court considered lis pendens, jurisdiction, and subpoena enforcement.
The plaintiff filed a motion concerning breach of contract and indemnification.
The legal brief discussed jurisdiction, discovery, deposition, and damages.
"""

In [ ]:
def load_corpus():
    # Each non-empty line becomes one training document.
    if CORPUS_FILE.exists():
        corpus_text = CORPUS_FILE.read_text(encoding="utf-8")
        print(f"Loaded corpus from: {CORPUS_FILE}")
        return corpus_text

    print(f"{CORPUS_FILE} was not found; using the built-in demo corpus.")
    return SAMPLE_CORPUS


def train_domain_tokenizer(corpus_text):
    # BPE repeatedly merges frequent neighboring units. Since it sees
    # domain vocabulary during training, it can learn compact pieces for
    # terms such as 'glioblastoma' and 'subpoena'.
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.normalizer = Sequence([NFD(), Lowercase(), StripAccents()])
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=DOMAIN_VOCAB_SIZE,
        min_frequency=1,
        special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    )
    documents = [line.strip() for line in corpus_text.splitlines() if line.strip()]
    tokenizer.train_from_iterator(documents, trainer=trainer)
    return tokenizer


def split_words(text):
    # This simple count is adequate for comparison in the demo.
    return re.findall(r"\b\w+\b", text)


def analyze_tokenizer(name, tokens, word_count):
    # Fertility is the average number of tokens per word. Lower is often
    # more efficient, but quality and general-language regression matter.
    token_count = len(tokens)
    fertility = token_count / word_count if word_count else 0
    attention_pairs = token_count ** 2
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Token count:           {token_count}")
    print(f"Word count:            {word_count}")
    print(f"Tokens per word:       {fertility:.2f}")
    print(f"Approximate n^2 work:  {attention_pairs:,}")
    return token_count


def estimate_attention_increase(base_token_count, domain_token_count):
    # Self-attention is approximately O(n^2). This estimates the ratio;
    # actual runtime also depends on batching and kernel optimizations.
    if base_token_count == 0:
        return 0
    length_ratio = domain_token_count / base_token_count
    cost_ratio = length_ratio ** 2
    print(f"\nSequence-length ratio: {length_ratio:.2f}x")
    print(f"Estimated attention-cost ratio: {cost_ratio:.2f}x")
    return cost_ratio

In [ ]:
corpus = load_corpus()

# In production, evaluate on a large held-out sample rather than one
# paragraph so the comparison does not overfit to these exact sentences.
evaluation_text = """
The patient was diagnosed with glioblastoma and aneurysm.
The legal dispute concerns subpoena enforcement and lis pendens.
"""

word_count = len(split_words(evaluation_text))
print(f"Evaluation words: {word_count}")

In [ ]:
# Load the general-purpose tokenizer and train the domain tokenizer.
general_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
general_tokens = general_tokenizer.tokenize(evaluation_text)
domain_tokenizer = train_domain_tokenizer(corpus)
domain_tokens = domain_tokenizer.encode(evaluation_text).tokens

print("General tokenizer pieces:")
print(general_tokens)
print("\nDomain tokenizer pieces:")
print(domain_tokens)

In [ ]:
general_count = analyze_tokenizer(
    "General-purpose tokenizer", general_tokens, word_count
)
domain_count = analyze_tokenizer(
    "Domain-specific tokenizer", domain_tokens, word_count
)

attention_ratio = estimate_attention_increase(general_count, domain_count)

print("\nInterpretation")
print("A domain tokenizer is useful when it reduces token count on the target corpus.")
print("Measure this on representative data before changing a production model.")

## Production caveats

This toy tokenizer is not ready for a model. Changing the vocabulary requires resizing and training the model's embedding and output layers, usually with continued pretraining. In practice, you might adapt the existing tokenizer instead of replacing it completely.

Evaluate token fertility, throughput, memory use, model quality, and regression on general-language tasks. The right answer is measured efficiency, not automatically choosing the tokenizer with the smallest vocabulary.